In [1]:
from pathlib import Path
import sys
exp_dir = Path.cwd().resolve().parent
if str(exp_dir) not in sys.path:
    sys.path.insert(0, str(exp_dir))
from embedding import LocalEmbeddings, BailianEmbeddings
import json
import numpy as np
from langchain_community.graphs.graph_document import GraphDocument, Node, Relationship
import math
from typing import Union, List, Dict, Any
import torch
import pickle
out_data_dir = "../../data/output_data/icews_yago"
source_dir = "../../data/source_data/icews_yago"
yago_kg_path = f"{source_dir}/yago_graph.pkl"
icews_kg_path = f"{source_dir}/icews_graph.pkl"

### Node 以及 Relationship 的字符串转换

In [ ]:
def get_node_signature(node: Node) ->str:
    properties = {}
    for k,v in node.properties.items():
        properties[k] = v
    property_str = ",".join(f"{k}={v}" for k, v in properties.items() if k != "name")
    node_type = node.type if node.type != "unknown" else "Entity"
    signature = f"{node_type}:{node.properties["name"]}({property_str})"
    return signature

def get_relationship_signature(relationship: Relationship) -> str:
    node1_signature = get_node_signature(relationship.source)
    node2_signature = get_node_signature(relationship.target)
    signature = f"{node1_signature}-[{relationship.type}]->{node2_signature}"
    return signature

def get_node_name(node: Node) -> str:
    signature = f"{node.properties['name']}"
    return signature

def get_node_id(node: Node) -> str:
    return node.id

### 构造 Node_id to Relationships 的映射关系

In [ ]:
kg_type = "icews"
# kg_type = "yago"

kg_path = yago_kg_path if kg_type == "yago" else icews_kg_path
with open(kg_path, "rb") as f:
    kg: GraphDocument = pickle.load(f)

node2rels = {}
for rel in kg.relationships:
    source_id = get_node_id(rel.source)
    target_id = get_node_id(rel.target)

    if source_id not in node2rels:
        node2rels[source_id] = []
    if target_id not in node2rels:
        node2rels[target_id] = []
    
    node2rels[source_id].append(rel)
    node2rels[target_id].append(rel)

"""
node2rels: Dict[str, List[Relationship]] = 
{
    node_id: [rel1, rel2, ...],
    ...
}
"""

#### 为每个节点利用Qwen的embedding API: text-embedding-v4 生成一个嵌入表示

In [ ]:
# kg_type = "yago"
kg_type = "icews"
node_embeddings = {}
node_signatures = []
for node in kg.nodes:
    signature = get_node_signature(node) # 是对节点签名做embedding
    node_signatures.append(signature)

embedding = BailianEmbeddings(api_key="sk-aec12d19a21049a2a56fa32acb855dae")
node_embs = embedding.embed_documents(node_signatures,2048) # 这里可以调整生成的embedding的维度

for node, emb in zip(kg.nodes, node_embs):
    id = get_node_id(node)
    node_embeddings[id] = emb
"""
node_embeddings: Dict[str, List[float]] = 
{
    node_id: embedding_vector,
    ...
}
"""

### 为每个节点挑选出一些特定的关系用于后续decoder-only模型的计算
#### 总体而言思路就是, 希望找到这样一些关系：其关联的邻居节点在另一个KG中有对应节点。

In [ ]:
# 为每个节点找出这样一些关系:
# 关系类型尽可能多样化
# 关联的邻居节点希望在另一个KG中有对应的节点, 也就是希望其邻居节点的embedding与另一个KG中某些节点的embedding相似
# 关联的邻居节点数量适中, 不要太多也不要太少
kg_type = "yago"
# kg_type = "icews"

# 加载另一个KG的节点embedding
other_kg_type = "icews" if kg_type == "yago" else "yago"
with open(f"{out_data_dir}/{other_kg_type}_node_embeddings(qwen).pkl", "rb") as f:
    other_node_embeddings: Dict[str, np.ndarray] = pickle.load(f)

# 先计算两个kg中节点embedding的相似度三角阵
node_ids = list(node_embeddings.keys())
other_node_ids = list(other_node_embeddings.keys())
node_emb_matrix = torch.tensor([node_embeddings[id] for id in node_ids])  # shape: (num_nodes, embedding_dim)
other_node_emb_matrix = torch.tensor([other_node_embeddings[id] for id in other_node_ids])  # shape: (num_other_nodes, embedding_dim)
import torch.nn.functional as F

# 先对两组 embedding 进行 L2 归一化
node_emb_norm = F.normalize(node_emb_matrix, p=2, dim=1)
other_node_emb_norm = F.normalize(other_node_emb_matrix, p=2, dim=1)
# [N, D] x [D, M] -> [N, M]
similarity_matrix = torch.mm(node_emb_norm, other_node_emb_norm.T) # 相似度矩阵，shape: (num_nodes, num_other_nodes)

top_k = 5
# 为当前kg中的每个节点找出在另一个kg中最相似的top-k个节点
top_k_values, top_k_indices = torch.topk(similarity_matrix, k=top_k, dim=-1)

node2topk = {}
for i, node_id in enumerate(node_ids):
    similar_nodes = []
    for j in range(top_k):
        score = top_k_values[i, j].item()
        other_id = other_node_ids[top_k_indices[i, j].item()]
        similar_nodes.append((other_id, score))
    node2topk[node_id] = similar_nodes
    
print(f"Extracted top-{top_k} similar nodes mappings.")


# 挑选规则参数
sim_threshold = 0.5       # 邻居节点在另一个KG中的最低相似度要求
max_rels_per_node = 15     # 控制关联邻居数目“适中”上限
max_same_type_rels = 3     # 同种类型最多挑选条数

selected_rels_for_node = {}

import tqdm
for node_id in tqdm.tqdm(node_ids):
    rels = node2rels[node_id]
    rel_candidates = []
    
    for rel in rels:
        src_id = get_node_id(rel.source)
        tgt_id = get_node_id(rel.target)
        neighbor_id = tgt_id if src_id == node_id else src_id
        
        # 查找邻居节点在另一张图里的最高相似度（因为上面已经排过序，所以取 [0][1] 即最大值）
        neighbor_max_sim = node2topk[neighbor_id][0][1] if neighbor_id in node2topk else 0.0
        rel_candidates.append((rel, neighbor_max_sim, rel.type, neighbor_id))
        
    # 按邻居节点的相似度降序排序，优先挑选邻居在匹配图中有高相似度的关系
    rel_candidates.sort(key=lambda x: x[1], reverse=True)
    
    picked_rels = []
    type_counter = {}  # 记录各类型被选中的次数
    
    for rel_info in rel_candidates:
        rel, sim, rel_type, neighbor_id = rel_info
        
        if sim < sim_threshold:
            continue
            
        current_type_count = type_counter.get(rel_type, 0)
        
        # 保证关系多样化：同类型的关系不要超过设定数量
        if current_type_count < max_same_type_rels:
            picked_rels.append(rel)
            type_counter[rel_type] = current_type_count + 1
            
        # 保证数量适中：不能太多
        if len(picked_rels) >= max_rels_per_node:
            break
            
    selected_rels_for_node[node_id] = picked_rels

"""
selected_rels_for_node: Dict[str, List[Relationship]] =
{
    node_id: [rel1, rel2, ...],  # 挑选出的关系列表
    ...
}
"""



### 利用上面生成的 Node embedding 做聚类
#### 聚类方式：K-means
* 为什么使用K-means：
* 此处我们的目的是：希望聚类后不仅得到各个节点的聚类结果，还希望为每个类别最后都得到一个可以代表该类的embedding，用途是：当我拿另一个kg中的节点的embedding与这些代表类的 embedding 做相似度比较时，可以保证与这个节点的 embedding 最相似的类中包含了与该节点语义相关的节点。
* 当在单位归一化的向量上使用K-Means时，完全模仿了通过余弦相似度/距离进行的聚类，因此，K-Means在这种情况下确实是基于余弦相似度进行聚类的。
* 根据定义，K-Means将一个节点分配给其最近的质心（即该类的代表嵌入）。因此，使用K-Means在理论上确实可以保证与未知节点语义相关的节点被聚类到跟未知节点最相似的中心点所在的类中。

In [ ]:
# 对节点签名做聚类，并且对于每个类别，获得一个可以表示该类的embedding
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
import numpy as np
import pickle

# kg_type = "icews"
kg_type = "yago"
node_ids = list(node_embeddings.keys())

kg_path = yago_kg_path if kg_type == "yago" else icews_kg_path

with open(kg_path, "rb") as f:
    kg: GraphDocument = pickle.load(f)
id_to_node = {get_node_id(n): n for n in kg.nodes}

X = np.array([node_embeddings[nid] for nid in node_ids]) # shape: (num_nodes, embedding_dim)

# 需要L2归一化来保证余弦相似度和欧式距离一致
X_norm = normalize(X)

num_clusters = 250 # 代表聚类后簇的数量

# 1. 训练KMeans获得质心向量 (代表类的embedding)
kmeans = KMeans(n_clusters=num_clusters, random_state=40) # random_state是为了保证结果可复现，设置一个固定的随机种子
kmeans.fit(X_norm)
centroids = kmeans.cluster_centers_ # shape: (num_clusters, embedding_dim)，每行是一个簇的质心向量，可以看作是该类的代表embedding

# 初始化保存聚类信息的列表
clusters_info = [{"id": i, "nodes": [], "embedding": centroids[i]} for i in range(num_clusters)]

# 引入“重叠聚类 (Overlapping Clustering)”
# 原先单纯的KMeans切分是硬边界，当一个未知节点落在两个簇的边界上时，它可能被分配到簇A，但它在图里最相近的那个点刚好越过了边界被分在簇B，这就可能导致匹配失败
# 解决方法：让图上的节点同时属于与它最相似的 Top-K 个簇。
# 这样边界节点会在周围几个簇中都存在(软聚类冗余)。
top_k_overlap = 2  # 每个节点会被归入最相似的k个簇中
# 计算所有节点到所有质心的余弦相似度
similarities = cosine_similarity(X_norm, centroids) # shape: (N, num_clusters)

# np.argsort默认升序，所以最后几个是相似度最大的
top_k_indices = np.argsort(similarities, axis=1)[:, -top_k_overlap:] # shape: (N, top_k_overlap)，每行是该节点最相似的top_k_overlap个簇的索引

for i, node_id in enumerate(node_ids):
    node = id_to_node[node_id]
    # 将该节点加入到对它来说最相似的 top-k 个簇中
    for cluster_id in top_k_indices[i]:
        clusters_info[cluster_id]["nodes"].append(node)

"""
[
    {
        "id": int, # 簇的ID
        "nodes": [Node(...), Node(...), ...],  # 属于这个簇的节点列表
        "embedding": np.array([...])  # 代表这个簇的质心向量
    },
    ...
]
"""

### 通过上面生成的聚类结果, 为每个节点找到一个对应的簇
#### 这里的思路是：对于 kg1 中的每个节点，计算它与 kg2 的每个簇的代表embedding之间的相似度，然后为这个节点分与其最相似的若干个簇。
#### 这里的相似度计算方式余弦相似度，与K-Means的聚类方式保持一致。

In [ ]:
# 根据embedding找到对应簇
kg_type1 = "icews"
kg_type2 = "yago"
cluster_mean = "KMeans_300clusters_3overlap_qwen"
from torch import Tensor
from torch import nn
import torch.nn.functional as F
with open(f"{out_data_dir}/{kg_type2}_clusters({cluster_mean}).pkl", "rb") as f:
    clusters_info_2 = pickle.load(f)
"""
[
    {
        "id": int,
        "nodes": list[Node],
        "embedding": numpy.ndarray
    },
   ...
]
"""

with open(f"{out_data_dir}/{kg_type1}_node_embeddings(qwen).pkl", "rb") as f:
    node_embeddings_1 = pickle.load(f)
"""
dict[node_id, embedding] = {
    "node_id":str,
    "embedding": list
"""

kg1_to_kg2_node_to_cluster = {}
cluster_num_for_each_node = 3 # 每个节点分配给最相似的3个簇
import tqdm
for node_id, embedding in tqdm.tqdm(node_embeddings_1.items()):
    # 计算当前节点和另一个kg中各个簇的相似度，将该节点分配给相似度最高的若干个簇
    assigned_cluster_id = [] # 可以分配给多个簇, 保存当前匹配到的簇id以及相似度
    embedding = Tensor(embedding)
    for cluster_info in clusters_info_2:
        cluster_embedding = Tensor(cluster_info["embedding"])
        sim = F.cosine_similarity(embedding, cluster_embedding, dim=0).item()
        if len(assigned_cluster_id) < cluster_num_for_each_node:
            assigned_cluster_id.append((cluster_info["id"], sim))
        else:
            # 如果当前簇的相似度比已分配簇中相似度最低的还高，则替换掉相似度最低的簇
            min_sim = min(assigned_cluster_id, key=lambda x: x[1])[1]
            if sim > min_sim:
                min_index = assigned_cluster_id.index(min(assigned_cluster_id, key=lambda x: x[1]))
                assigned_cluster_id[min_index] = (cluster_info["id"], sim)
    # 最后按照相似度从高到低排序
    kg1_to_kg2_node_to_cluster[node_id] = sorted(assigned_cluster_id, key=lambda x: x[1], reverse=True)
    kg1_to_kg2_node_to_cluster[node_id] = [cluster_id for cluster_id, sim in kg1_to_kg2_node_to_cluster[node_id]]
    # break # 先测试一个节点
# 保存kg1_to_kg2_node_to_cluster
with open(f"{out_data_dir}/{kg_type1}_to_{kg_type2}_node_to_cluster(qwen).pkl", "wb") as f:
    pickle.dump(kg1_to_kg2_node_to_cluster, f)
"""
dict[node_id, cluster_id] = {
    node_id: str,
    cluster_id: list[cluster_id]
}
"""


#### 加载ref_pair信息，查看对于 kg1 中的每个节点，为其分配的簇中是否存在ref_pair中与其对应的节点。
* 计算匹配的准确率，同时拿到所有匹配到的节点的id

In [ ]:
kg_type1 = "icews"
kg_type2 = "yago"
with open(f"{out_data_dir}/{kg_type2}_clusters({cluster_mean}).pkl", "rb") as f:
    clusters_info_2 = pickle.load(f) # kg2的聚类信息
"""
{
    "id": int,
    "nodes": list[Node],
    "embedding": numpy.ndarray
}
"""
with open(f"{out_data_dir}/{kg_type1}_to_{kg_type2}_node_to_cluster(qwen).pkl", "rb") as f:
    kg1_to_kg2_node_to_cluster = pickle.load(f) # kg1中每个节点对应的kg2中簇的列表
"""
{
    node_id: list[cluster_id]
}
"""
graph_dir = "../../data/source_data/icews_yago"
with open(f"{graph_dir}/icews2yago.json", "r") as f:
    icews_id2yago_id = json.load(f) # icews中节点id到yago中节点id的映射关系
"""
{
    icews_node_id: yago_node_id,
    ...
}
"""

# 构造一个kg1的id to node
with open(f"{graph_dir}/{kg_type1}_graph.pkl", "rb") as f:
    kg1:GraphDocument = pickle.load(f)
id2node = {}
for node in kg1.nodes:
    id2node[node.id] = node

node_num = len(icews_id2yago_id)
matched_nodes_id = set()
for node_id, cluster_id_list in kg1_to_kg2_node_to_cluster.items():
    matched_id = icews_id2yago_id.get(node_id)
    if matched_id is None:
        continue
    for i in range(len(cluster_id_list)):
        cluster_id, sim = cluster_id_list[i]
        cluster_info = {}
        for cluster_ in clusters_info_2:
            if cluster_["id"] == cluster_id:
                cluster_info = cluster_
                break
        cluster_nodes:List[Node] = cluster_info["nodes"]
        same_name_nodes = [node for node in cluster_nodes if node.id == matched_id]
        if same_name_nodes:
            matched_nodes_id.add(node_id)
    
print(f"Total nodes: {node_num}, Matched nodes: {len(matched_nodes_id)}, Matching ratio: {len(matched_nodes_id)/node_num:.4f}")

* 对于匹配到的节点，构造一个node_id to list(node_id)的映射关系，即将匹配到的簇中的节点拿出来，便于后续用decoder-only/sentence-embedding等方式计算。

In [ ]:
# 从matched_nodes_id构造matched_nodes_id_to_nodes
matched_nodes_id_to_node = {}
"""
{
    node_id: set(node_id, node_id, ...) # 匹配的簇中包含的节点id集合
}
"""
for node_id in matched_nodes_id:
    cluster_info = kg1_to_kg2_node_to_cluster[node_id]
    cluster_ids = [cluster_id for cluster_id, sim in cluster_info]
    matched_nodes_id_to_node[node_id] = set()
    for cluster_id in cluster_ids:
        cluster_info = {}
        for cluster_ in clusters_info_2:
            if cluster_["id"] == cluster_id:
                cluster_info = cluster_
                break
        cluster_nodes:List[Node] = cluster_info["nodes"]
        for node in cluster_nodes:
            matched_nodes_id_to_node[node_id].add(node.id)
# 保存matched_nodes_id_to_node
with open(f"{out_data_dir}/{kg_type1}_matched_nodes_id_to_nodes(qwen).pkl", "wb") as f:
    pickle.dump(matched_nodes_id_to_node, f)